In [1]:
from pydantic_ai import Agent
from toyaikit.tools import get_instance_methods

In [2]:
from gitsource import GithubRepositoryDataReader
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

parsed_docs = [doc.parse() for doc in files]

index = AppendableIndex(
    text_fields=["title", "description", "content", "filename"],
    keyword_fields=[]
)

index.fit(parsed_docs)

In [3]:
from minsearch import Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS

stopwords = DEFAULT_ENGLISH_STOP_WORDS | {"evidently"}

highlighter = Highlighter(
    highlight_fields=["content"],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(
        stemmer="snowball",
        stop_words=stopwords
    )
)

In [4]:
file_index = {
    doc["filename"]: doc["content"]
    for doc in parsed_docs
}

In [5]:
from typing import Any, Dict, List


class SearchTools:
    """
    Provides search and file retrieval utilities over an indexed data store.
    """

    def __init__(
        self,
        index: Any,
        highlighter: Any,
        file_index: Dict[str, str]
    ) -> None:
        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the documentation and return relevant snippets.

        Args:
            query: The search query.
        """
        search_results = self.index.search(
            query=query,
            num_results=5
        )

        return self.highlighter.highlight(
            query,
            search_results
        )

    def get_file(self, filename: str) -> str:
        """
        Return the full content of a documentation file.

        Args:
            filename: The filename to retrieve.
        """
        if filename in self.file_index:
            return self.file_index[filename]

        return f"File {filename} does not exist."

In [6]:
search_tools = SearchTools(
    index,
    highlighter,
    file_index
)

tools = get_instance_methods(search_tools)

tools

[<bound method SearchTools.get_file of <__main__.SearchTools object at 0x71adb5393770>>,
 <bound method SearchTools.search of <__main__.SearchTools object at 0x71adb5393770>>]

In [7]:
instructions = """
You're a documentation assistant.

Answer the user question using the documentation knowledge base.

First, search the documentation for relevant information.

Analyze the search results and identify the most relevant files.

Use get_file to retrieve the full content of the relevant files.

Read and analyze the full files before producing the final answer.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to
include the word 'evidently' in search queries.
"""

search_agent = Agent(
    name="search",
    model="openai:gpt-4o-mini",
    instructions=instructions,
    tools=tools
)

In [8]:
query = "How do I create an Evidently dashboard?"

result = await search_agent.run(query)

In [9]:
result.output

'To create a dashboard in Evidently, follow these steps:\n\n### 1. **Set Up Your Project**\n- Ensure you are connected to **Evidently Cloud**, and create a new Project where your data evaluations will be logged.\n\n### 2. **Access the Dashboard**\n- Each Project has its own Dashboard, initially empty. You first need to run evaluations and save at least one Report in your Project.\n\n### 3. **Adding Tabs (Optional)**\n- In the Dashboard, you can organize information into multiple Tabs. \n  - To add a new Tab: \n    - Enter **Edit** mode.\n    - Click the "+" sign to add a Tab.\n    - Name your Tab (you can also select templates for pre-built tabs).\n\n### 4. **Adding Panels**\nYou can add various types of Panels to visualize your data. Panels can include counters, line plots, bar plots, and more. Here’s how to add a Panel:\n\n#### **Using the UI:**\n- While in **Edit** mode, click **Add Panel**.\n- Configure your Panel:\n  - **Select Metrics**: Choose the metric from your Reports.\n  - 

In [10]:
result.all_messages()

[ModelRequest(parts=[UserPromptPart(content='How do I create an Evidently dashboard?', timestamp=datetime.datetime(2026, 10, 10, 10, 56, 37, 198248, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 10, 10, 10, 56, 37, 198625, tzinfo=datetime.timezone.utc), instructions="You're a documentation assistant.\n\nAnswer the user question using the documentation knowledge base.\n\nFirst, search the documentation for relevant information.\n\nAnalyze the search results and identify the most relevant files.\n\nUse get_file to retrieve the full content of the relevant files.\n\nRead and analyze the full files before producing the final answer.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to\ninclude the word 'evidently' in search queries.", run_id='01a12575-2736-7485-b0a3-34455cd4d4cc', conversation_id='01a12575-2736-7485-b0a3-34468a34fbd2'),
 ModelRe

In [11]:
result.new_messages()

[ModelRequest(parts=[UserPromptPart(content='How do I create an Evidently dashboard?', timestamp=datetime.datetime(2026, 10, 10, 10, 56, 37, 198248, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 10, 10, 10, 56, 37, 198625, tzinfo=datetime.timezone.utc), instructions="You're a documentation assistant.\n\nAnswer the user question using the documentation knowledge base.\n\nFirst, search the documentation for relevant information.\n\nAnalyze the search results and identify the most relevant files.\n\nUse get_file to retrieve the full content of the relevant files.\n\nRead and analyze the full files before producing the final answer.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to\ninclude the word 'evidently' in search queries.", run_id='01a12575-2736-7485-b0a3-34455cd4d4cc', conversation_id='01a12575-2736-7485-b0a3-34468a34fbd2'),
 ModelRe

In [12]:
result2 = await search_agent.run(
    "Show me the Python code",
    message_history=result.all_messages()
)

In [13]:
result2.output

'Here’s an example of Python code to create a Dashboard in Evidently, including adding Tabs and Panels:\n\n### Example Code\n\n```python\nfrom evidently.sdk.models import PanelMetric\nfrom evidently.sdk.panels import DashboardPanelPlot\nfrom evidently.sdk import Project  # Make sure to import the Project from the Evidently SDK\n\n# Connect to your Evidently Project\nproject = Project("your_project_name_here")\n\n# Create a new Dashboard Tab\nproject.dashboard.add_tab("Evaluation Results")\n\n# Adding a Counter Panel for Total Row Count\nproject.dashboard.add_panel(\n    DashboardPanelPlot(\n        title="Total Row Count",\n        subtitle="Total evaluations over time.",\n        size="half",\n        values=[\n            PanelMetric(legend="Row count", metric="RowCount")\n        ],\n        plot_params={"plot_type": "counter", "aggregation": "sum"},\n    ),\n    tab="Evaluation Results"\n)\n\n# Adding a Line Chart Panel for Row Count\nproject.dashboard.add_panel(\n    DashboardPane

In [15]:
result.usage

RunUsage(cost=Decimal('0.00123135'), details={'reasoning_tokens': 0}, input_tokens=5689, output_tokens=630, requests=3, tool_calls=4)

In [16]:
from pydantic_ai import RunUsage

In [17]:
messages = []
usage = RunUsage()

while True:
    user_prompt = input("You:")

    if user_prompt.lower().strip() == "stop":
        break

In [18]:
messages = []
usage = RunUsage()

while True:
    user_prompt = input("You:")

    if user_prompt.lower().strip() == "stop":
        break

    result = await search_agent.run(
        user_prompt,
        message_history=messages
    )

    usage = usage + result.usage
    messages = result.all_messages()

    print("Assistant:", result.output)

Assistant: To create a dashboard, follow these steps:

### Overview
A **Dashboard** provides a visual representation of your AI application performance, allowing you to track evaluation results and production quality over time. Each project comes with a dashboard that is initially empty.

### Prerequisites
1. **Create a Project**: Ensure that you have created a project in either Evidently Cloud, OSS, or Enterprise.
2. **Add Reports**: Populate the dashboard by saving at least one report with evaluation results in the project.

### Creating a Dashboard

#### 1. **Accessing the Dashboard**
- Log in to your project on Evidently.

#### 2. **Adding Tabs (Optional)**
- Enter **Edit** mode on the dashboard.
- Click the plus sign to add a new tab, or create an empty one with a custom name. You can also use pre-built tabs.

#### 3. **Adding Panels**
You can add panels either from the user interface or programmatically using Python. Here’s how to do both:

**Using the User Interface**:
- In **Ed

In [19]:
usage

RunUsage(cost=Decimal('0.00126255'), details={'reasoning_tokens': 0}, input_tokens=5685, output_tokens=683, requests=3, tool_calls=4)

In [20]:
from pydantic_ai.messages import FunctionToolCallEvent

In [21]:
class NamedCallback:

    def __init__(self, agent):
        self.agent_name = agent.name

    async def print_function_calls(self, ctx, event):
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)

In [22]:
callback = NamedCallback(search_agent)

result = await search_agent.run(
    "how do I build dashboards with evidently?",
    event_stream_handler=callback
)

TOOL CALL (search): search({"query":"build dashboards"})
TOOL CALL (search): get_file({"filename":"docs/platform/dashboard_panel_types.mdx"})
TOOL CALL (search): get_file({"filename":"examples/LLM_regression_testing.mdx"})
